# Stream 3 - Clinical (maternal) stillbirth-risk: leakage audit and final model

**Why this notebook exists.** The first Stream 3 model reported AUC 0.9684. The audit below shows
that number came from gestational age at delivery (COMBGEST), which is decided by the outcome, and
from "stillbirths" under 20 weeks. The final model uses only pregnancies at 20 weeks or later,
no outcome-time variables, and a temporal split (train 2016-2021, test 2022-2023).

**Data caveat.** Stillbirths and live births come from different CDC files, and live births were
sampled (about 50/50). AUC is usable; precision, recall and PR-AUC are NOT clinical numbers.
Absolute risk needs the prior-shift correction in section 5.

In [ ]:
import os, json, warnings
import numpy as np, pandas as pd, xgboost as xgb, joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive not mounted (fine outside Colab):", e)

CSV = os.environ.get("S3_CSV", "/content/drive/MyDrive/fetal-health-project/data/processed/clinical_training_data.csv")
OUT = os.environ.get("S3_OUT", "/content/drive/MyDrive/fetal-health-project/models/stream3_v2")
N_EST = int(os.environ.get("S3_NEST", "300"))
RUN_FULL_AUDIT = os.environ.get("S3_AUDIT", "1") == "1"   # False = skip the audit table, run only the final model
N_BOOT = int(os.environ.get("S3_BOOT", "200"))
SEED = 42
# PLACEHOLDER: approximate real-world stillbirth prevalence (20+ weeks). VERIFY against CDC/NVSS before the paper.
TRUE_PREVALENCE = 0.006
os.makedirs(OUT, exist_ok=True)

NUM_FEATURES = ["MAGER", "BMI", "PRECARE", "PRIORDEAD", "PRIORLIVE", "MEDUC"]
CAT_FEATURES = ["RF_GDIAB", "RF_GHYPE", "RF_EHYPE", "RF_CESAR", "RF_ARTEC", "RF_INFTR", "RF_FEDRG",
                "CIG_REC", "WIC", "DPLURAL", "SEX", "MRACE6", "MBSTATE_REC"]
UNKNOWN_CODES = ["U", "9", "99", "Unknown"]
# variables that are only known at/after delivery -> must never be model inputs
OUTCOME_TIME_VARS = ["COMBGEST", "OEGest_Comb", "OBGEST_FLG", "DBWT", "BWTR4"]
assert not set(NUM_FEATURES + CAT_FEATURES) & set(OUTCOME_TIME_VARS), "outcome-time variable used as a feature!"


## 1. Load and clean (same cleaning as the original notebook)

In [ ]:
raw = pd.read_csv(CSV, low_memory=False)
for c in CAT_FEATURES:
    raw[c] = raw[c].replace(UNKNOWN_CODES, np.nan)
raw["COMBGEST"] = raw["COMBGEST"].replace([99], np.nan)
df = raw.drop_duplicates().reset_index(drop=True)      # deduplicated copy used for everything except audit rows A-D
print("raw:", raw.shape, "| deduplicated:", df.shape, "| duplicates removed:", len(raw) - len(df))
print("outcome counts (dedup):", df.OUTCOME.value_counts().to_dict())
bins = [0, 12, 20, 28, 37, 50]
print(pd.crosstab(pd.cut(raw.COMBGEST, bins, right=False), raw.OUTCOME, dropna=False))
print("\n'Stillbirths' under 20 weeks (not stillbirths by the usual definition): "
      f"{int(((raw.OUTCOME == 1) & (raw.COMBGEST < 20)).sum())} of {int((raw.OUTCOME == 1).sum())}")


## 2. Helpers

In [ ]:
def make_pipe(num, cat):
    parts = [("n", SimpleImputer(strategy="median"), num)]
    if cat:
        parts.append(("c", Pipeline([("i", SimpleImputer(strategy="most_frequent")),
                                     ("o", OneHotEncoder(handle_unknown="ignore"))]), cat))
    model = xgb.XGBClassifier(n_estimators=N_EST, max_depth=5, learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, eval_metric="logloss", random_state=SEED,
                              tree_method="hist", n_jobs=-1)
    return Pipeline([("pre", ColumnTransformer(parts)), ("m", model)])

def fit_eval(d, num, cat, mode="random"):
    """mode='random': stratified 80/20 (seed 42). mode='temporal': train YEAR<=2021, test YEAR>=2022."""
    if mode == "temporal":
        tr, te = d[d.YEAR <= 2021], d[d.YEAR >= 2022]
    else:
        tr, te = train_test_split(d, test_size=0.2, random_state=SEED, stratify=d.OUTCOME)
    pipe = make_pipe(num, cat).fit(tr[num + cat], tr.OUTCOME)
    p = pipe.predict_proba(te[num + cat])[:, 1]
    return pipe, tr, te, p

audit = []
def record(name, d, num, cat, mode="random"):
    _, tr, te, p = fit_eval(d, num, cat, mode)
    row = dict(setting=name, split=mode, n=len(d), stillbirths=int(d.OUTCOME.sum()),
               auc=round(roc_auc_score(te.OUTCOME, p), 4), pr_auc_not_clinical=round(average_precision_score(te.OUTCOME, p), 4))
    audit.append(row)
    print(f"{name:52s} [{mode:8s}] n={row['n']:7d} AUC={row['auc']:.4f}")


## 3. Leakage audit (rows A-D use the non-deduplicated data so the original 0.9684 is reproduced)

In [ ]:
if RUN_FULL_AUDIT:
    raw20 = raw[raw.COMBGEST >= 20]
    record("A all rows, WITH COMBGEST (original 0.9684)", raw, NUM_FEATURES + ["COMBGEST"], CAT_FEATURES)
    record("B all rows, no COMBGEST", raw, NUM_FEATURES, CAT_FEATURES)
    record("C >=20 wk, WITH COMBGEST", raw20, NUM_FEATURES + ["COMBGEST"], CAT_FEATURES)
    record("D >=20 wk, no COMBGEST", raw20, NUM_FEATURES, CAT_FEATURES)
    d20 = df[df.COMBGEST >= 20].copy()
    record("D2 >=20 wk dedup, minus PRECARE", d20, [c for c in NUM_FEATURES if c != "PRECARE"], CAT_FEATURES)
    cc = d20.dropna(subset=NUM_FEATURES + CAT_FEATURES)
    record("D3 >=20 wk, complete cases only", cc, NUM_FEATURES, CAT_FEATURES)
    record("D4 >=37 wk, complete cases only", cc[cc.COMBGEST >= 37], NUM_FEATURES, CAT_FEATURES)
    record("D5 >=20 wk, numeric features only", d20, NUM_FEATURES, [])
    pd.DataFrame(audit).to_csv(os.path.join(OUT, "audit_table.csv"), index=False)


## 4. FINAL model: >=20 weeks, no outcome-time variables, temporal split

In [ ]:
d20 = df[df.COMBGEST >= 20].copy()
FEATS = NUM_FEATURES + CAT_FEATURES
print("features used:", FEATS)
pipe, tr, te, p = fit_eval(d20, NUM_FEATURES, CAT_FEATURES, "temporal")
y = te.OUTCOME.values
auc = roc_auc_score(y, p)
rng = np.random.RandomState(0)
boots = []
for _ in range(N_BOOT):
    i = rng.randint(0, len(y), len(y))
    if y[i].min() != y[i].max():
        boots.append(roc_auc_score(y[i], p[i]))
lo, hi = np.percentile(boots, [2.5, 97.5])
print(f"FINAL temporal test (train<=2021 n={len(tr)}, test>=2022 n={len(te)}): AUC={auc:.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

band_rows = []
for a, b in [(20, 28), (28, 37), (37, 50)]:
    m = ((te.COMBGEST >= a) & (te.COMBGEST < b)).values
    if m.sum() > 50 and 0 < y[m].mean() < 1:
        band_rows.append(dict(band=f"{a}-{b} wk", n=int(m.sum()), stillbirth_share=round(float(y[m].mean()), 3),
                              auc=round(roc_auc_score(y[m], p[m]), 4)))
print(pd.DataFrame(band_rows))


## 5. Calibration and absolute risk (prior-shift correction)
The sample is not the real population (live births were sampled). Raw probabilities are calibrated to the
*sample*. To show absolute risk, odds are rescaled from the sample prevalence to TRUE_PREVALENCE
(assumes the sampled live births are representative; TRUE_PREVALENCE is a placeholder to verify).

In [ ]:
pi_train = float(tr.OUTCOME.mean())
odds = p / np.clip(1 - p, 1e-9, None)
odds_c = odds * (TRUE_PREVALENCE / (1 - TRUE_PREVALENCE)) / (pi_train / (1 - pi_train))
p_corr = odds_c / (1 + odds_c)
print(f"sample prevalence (train) {pi_train:.3f} -> assumed real prevalence {TRUE_PREVALENCE:.4f}")
print(f"Brier score on the sample: {brier_score_loss(y, p):.4f}")

frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
plt.figure(figsize=(4.5, 4.5))
plt.plot([0, 1], [0, 1], "--", color="gray", label="perfect")
plt.plot(mean_pred, frac_pos, "o-", label="model (test 2022-23)")
plt.xlabel("predicted risk (sample)"); plt.ylabel("observed stillbirth share (sample)")
plt.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUT, "calibration.png"), dpi=150); plt.show()

dec = pd.qcut(p, 10, labels=False, duplicates="drop")
tab = pd.DataFrame({"decile": dec, "sample_rate": y, "corrected_risk": p_corr}).groupby("decile").mean().round(4)
tab["n"] = pd.Series(dec).value_counts().sort_index().values
print("\nrisk by decile (0 = lowest risk):"); print(tab)


## 6. Explainability: mean |SHAP| per original feature (XGBoost built-in contributions, no extra library)

In [ ]:
pre, booster = pipe.named_steps["pre"], pipe.named_steps["m"].get_booster()
n_sub = min(5000, len(te))
Xs = pre.transform(te[FEATS].iloc[:n_sub])
contrib = booster.predict(xgb.DMatrix(Xs), pred_contribs=True)[:, :-1]
names = [n.split("__", 1)[1] for n in pre.get_feature_names_out()]
def group_of(n):
    for c in sorted(FEATS, key=len, reverse=True):
        if n == c or n.startswith(c + "_"):
            return c
    return n
groups = pd.Series(np.abs(contrib).mean(0), index=[group_of(n) for n in names]).groupby(level=0).sum().sort_values()
groups.plot.barh(figsize=(6, 5), title="Mean |SHAP| per feature (stillbirth risk)")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "shap_importance.png"), dpi=150); plt.show()
print(groups.sort_values(ascending=False).round(4))


## 7. Save (new folder; the old stream3_clinical_xgb.json is NOT overwritten)

In [ ]:
pipe.named_steps["m"].save_model(os.path.join(OUT, "stream3_clinical_xgb_v2.json"))
joblib.dump(pipe, os.path.join(OUT, "stream3_pipeline_v2.joblib"))
pd.DataFrame({"year": te.YEAR.values, "outcome": y, "p_sample": p, "p_corrected": p_corr}).to_csv(
    os.path.join(OUT, "temporal_test_predictions.csv"), index=False)
with open(os.path.join(OUT, "results_v2.json"), "w") as f:
    json.dump(dict(features=FEATS, scope=">=20 weeks, COMBGEST not a feature, temporal split 2016-2021 / 2022-2023",
                   n_train=len(tr), n_test=len(te), auc=round(float(auc), 4), auc_ci95=[round(float(lo), 4), round(float(hi), 4)],
                   bands=band_rows, true_prevalence_assumed=TRUE_PREVALENCE, audit=audit), f, indent=1)
print("saved to", OUT)
